# 상품평 텍스트 수집 및 전처리 (엑셀 3종)

**환경:** Google Colab / Jupyter Notebook
**핵심 도구:** pandas · re · scikit-learn · kiwipiepy
**결과물:** `cleaned_<엑셀파일명>.csv` (단원 3에서 사용)

## 이번 실습에서 쓰는 엑셀 파일 (3가지 중 하나 선택)
| 파일 | 카테고리 | 리뷰 수 | 이런 리뷰가 많아요 |
|---|---|---|---|
| `TS_5-4.청소세탁용품.xlsx` | 청소/세탁용품 | 4,401 | 건조대, 변기솔, 조립, 튼튼함 |
| `TS_4-4.자동차기기.xlsx` | 자동차기기 | 8,001 | 블랙박스, 내비, 화질, 설치 |
| `TS_3-1.영상음향가전.xlsx` | 영상/음향가전 | 6,001 | TV, 사운드바, 음질, 화질 |

세 파일은 **열 구성이 같고**(`INDEX`, `상품평` 등), **리뷰에 쓰이는 단어가 다릅니다.**
그래서 코드는 하나로 쓰고, **파일마다 다른 부분만 `설정`으로 바꿉니다.**

## 학습 목표
1. 엑셀 상품평을 불러와 빈 값·중복을 점검한다.
2. 문장, 띄어쓰기 단어, scikit-learn 토큰의 차이를 비교한다.
3. 정규표현식으로 개인정보를 가리고 특수문자를 정리한다.
4. **데이터에 맞는** 표기 통일 규칙과 형태소 분석 설정을 적용한다.
5. Kiwi 형태소 분석 결과를 scikit-learn에 연결하고 CSV로 저장한다.

## 실습 진행 과정
| 단계 | 할 일 | 확인할 결과 |
|---|---|---|
| 1. 준비 | 설치 → 엑셀 업로드 → **파일별 설정 자동 선택** → 데이터 점검 | 선택된 설정 이름, 리뷰 수 |
| 2. 문장·단어 | 첫 리뷰로 문장 나누기, 띄어쓰기 나누기, scikit-learn 토큰 비교 | 한 글자 단어가 빠지는 문제 |
| 3. 정규식 정제 | 이메일·전화번호 가리기 → 특수문자·공백 정리 | 정제텍스트 |
| 4. 표기 통일 | **공통 규칙 + 파일별 규칙** 적용 | 바뀐 리뷰 수와 예시 |
| 5. 형태소 분석 | **파일별 사용자 사전** 등록 → 필요한 단어 고르기 → 자주 나온 단어 확인 | 분석토큰, 상위 단어 |
| 6. 저장 | 결과 CSV 저장 | `cleaned_<파일명>.csv` |

> **실행 방법:** 셀을 클릭하고 `Shift + Enter`. 반드시 **위에서부터 순서대로** 실행하세요.
> 다른 엑셀 파일로 다시 하려면 **1-2 업로드 셀부터** 다시 실행하면 됩니다.

---
## 1. 환경 준비 및 데이터 확인

### 1-1. 설치와 불러오기

In [1]:
# 한국어 형태소 분석기와 엑셀 읽기 도구 설치 (처음 한 번만)
!pip install -q kiwipiepy openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 9.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 43.9 MB/s eta 0:00:00


In [2]:
# 표준 라이브러리 불러오기
import importlib.util  # 모듈 설치 여부 확인
import subprocess      # pip 명령 실행
import sys             # 현재 파이썬 실행 경로 확인


# 실습에 사용할 라이브러리 불러오기
import pandas as pd                                          # 표 형태 데이터 처리
import re                                                    # 정규표현식
from sklearn.feature_extraction.text import CountVectorizer  # 토큰화·단어 빈도
from kiwipiepy import Kiwi                                   # 한국어 형태소 분석기

# 설치 결과 확인 출력
print("pandas:", pd.__version__)
print("실습 환경 준비 완료")

pandas: 2.2.3
실습 환경 준비 완료


### 1-2. 엑셀 업로드
`엑셀파일명.csv`를 Colab 왼쪽 **왼쪽 파일 창(📁)에 끌어다 놓으면** 그 데이터를 씁니다.

In [3]:
import glob

# ===== ✏️ [입력] 찾을 엑셀 파일 이름 (앞부분만 써도 됨, *는 '아무 글자') =====
excel_path = glob.glob("TS_*.xlsx")[0]     # [0]: 반환된 리스트의 첫 번째 경로를 꺼냄
print(excel_path)

TS_3-1.영상음향가전.xlsx


### 1-3. ✏️ 엑셀 파일마다 바뀌는 부분 (설정)

#### 왜 파일마다 설정이 달라야 할까?
같은 코드라도 **리뷰에 나오는 단어가 다르기 때문**입니다. 실제 데이터를 분석해서 찾은 차이입니다.

| 설정 | 하는 일 | 청소/세탁용품 (TS_5-4) | 자동차기기 (TS_4-4) | 영상/음향가전 (TS_3-1) |
|---|---|---|---|---|
| `replace_rules` | 틀린 표기·줄임말을 하나로 통일 | `프라스틱`→`플라스틱` (7건)<br>`크리너`→`클리너` (16건)<br>`헹거`→`행거`, `배란다`→`베란다` | `네비`→`내비` (620건)<br>`블박`→`블랙박스` (127건)<br>`시가잭`→`시거잭` (21건)<br>`업뎃`→`업데이트`, `스팩`→`스펙` | `티비`→`TV` (650건)<br>+ 영어 대문자 통일<br>(`tv`, `hdmi`, `usb` → `TV`, `HDMI`, `USB`) |
| `user_words` | 분석기가 **잘못 나누는 단어**를 한 단어로 등록 | `변기솔` (변기+솔)<br>`청소솔`, `연마제`, `롱핸들` | `길안내` (길+안내)<br>`앞유리`, `후방카메라`, `구성품` | `사운드바` (사운드+바)<br>`미러링`, `베젤`, `중저음` |
| `stop_words` | 이 데이터에서 너무 흔해 **구별에 도움이 안 되는 단어** | `빨래`, `청소` | `차` | `티비`, `TV` |

**모든 파일에 공통으로 적용하는 규칙**
- 익명 처리된 브랜드명 `OO`, `○○` 지우기 (자동차기기 600건, 영상/음향가전 228건)
- `스크레치`→`스크래치`, `재 구매`→`재구매`, `읍니다`→`습니다`, 띄어 쓴 `넘`→`너무`
- 불용어: `사용`, `제품`, `상품`, `구매`, `구입`, `생각`, `하다`, `되다`, `있다`, `같다`

> 아래 셀은 **업로드한 파일 이름을 보고 설정을 자동으로 고릅니다.** 파일 이름이 달라 자동 선택이 안 되면 `DATA_KEY`에 직접 적으세요.

In [4]:
# ================= ✏️ [수정 가능] 엑셀 파일별 설정 =================
SETTINGS = {
    "TS_5-4": {   # 청소/세탁용품
        "이름": "청소/세탁용품",
        "replace_rules": {"프라스틱": "플라스틱", "크리너": "클리너", "헹거": "행거", "배란다": "베란다", "스텐레스": "스테인리스"},
        "user_words": ["변기솔", "청소솔", "연마제", "롱핸들", "올스텐", "빨래건조대", "유리닦이"],
        "stop_words": {"빨래", "청소"},
    },
    "TS_4-4": {   # 자동차기기
        "이름": "자동차기기",
        "replace_rules": {"네비": "내비", "블박": "블랙박스", "시가잭": "시거잭", "업뎃": "업데이트", "스팩": "스펙", "퀼리티": "퀄리티"},
        "user_words": ["길안내", "앞유리", "후방카메라", "구성품", "매립형", "시거잭", "블랙박스"],
        "stop_words": {"차"},
    },
    "TS_3-1": {   # 영상/음향가전
        "이름": "영상/음향가전",
        "replace_rules": {"티비": "TV"},
        "user_words": ["사운드바", "미러링", "베젤", "중저음", "리어스피커", "빔프로젝터", "돌비애트모스"],
        "stop_words": {"TV"},
    },
}

# 모든 파일에 공통으로 쓰는 규칙
COMMON = {
    "replace_rules": {"스크레치": "스크래치", "재 구매": "재구매", "읍니다": "습니다"},
    "stop_words": {"사용", "제품", "상품", "구매", "구입", "생각", "하다", "되다", "있다", "같다"},
}

# 자동 선택이 안 될 때 직접 고르기 ("TS_5-4", "TS_4-4", "TS_3-1" 중 하나, 자동이면 None)
DATA_KEY = None
# =====================================================================

# 파일 이름에 들어 있는 키(TS_5-4 등)로 설정 자동 선택
if DATA_KEY is None:
    DATA_KEY = next((key for key in SETTINGS if key in excel_path), None)

# 설정을 찾으면 그 설정, 못 찾으면 빈 설정(공통 규칙만 사용)
if DATA_KEY in SETTINGS:
    cfg = SETTINGS[DATA_KEY]
    print("선택된 설정:", DATA_KEY, "-", cfg["이름"])
else:
    cfg = {"이름": "기타", "replace_rules": {}, "user_words": [], "stop_words": set()}
    print("[안내] 파일별 설정을 찾지 못해 공통 규칙만 사용합니다. DATA_KEY를 직접 적어도 됩니다.")

선택된 설정: TS_3-1 - 영상/음향가전


### 1-4. 엑셀 읽기와 데이터 점검

In [5]:
# 필요한 열(INDEX, 상품평)만 문자열로 읽기
df = pd.read_excel(excel_path, usecols=["INDEX", "상품평"], dtype=str)

# 이후 코드에서 쓰기 쉽게 열 이름 바꾸기
df = df.rename(columns={"INDEX": "번호", "상품평": "원본텍스트"})

# 빈 값과 중복 개수 확인
print("리뷰 수:", len(df))
print("빈 값 수:", df["원본텍스트"].isna().sum())
print("중복 수:", df["원본텍스트"].duplicated().sum())

# 빈 값·중복을 뺀 작업용 복사본 만들기 (원본 df는 그대로 둠)
work = df.dropna(subset=["원본텍스트"]).drop_duplicates(subset=["원본텍스트"]).copy()
print("정리 후 리뷰 수:", len(work))

# 앞부분 5개 보기
work.head()

리뷰 수: 6001
빈 값 수: 0
중복 수: 0
정리 후 리뷰 수: 6001


,번호,원본텍스트
0,2,기존 보유중인 55인치보다 작네요 사이즈 무슨일인가요..?;;;;\n블루투스 작동 ...
1,3,크기가 55인치라 거실에 두고 사용하니 온 가족이 볼 수 있게 딱 적당하네요~\n사...
2,4,해피콜부터 배송이 그렇게 빠른건아니지만 설치기사님들 친절하시고\n설명잘해주시고 가셔...
3,5,방송보고 주문한지 꽤 됐는데 공사일정으로 이제 설치 받았어요~\n설치기사님이 너무너...
4,6,55인치쓰다가 65인치로바꾸니 좀 많이커요 ㅎㅎㅎ\n그리고 매뉴에서 각종 어플싸이크...


---
## 2. 문장·단어 단위 처리

| 방법 | 기준 | 특징 |
|---|---|---|
| 문장 나누기 | `.` `!` `?` 뒤 공백 | 마침표 없는 리뷰는 나뉘지 않음 (간단한 규칙) |
| 띄어쓰기 나누기 | 공백 | 조사·어미가 단어에 붙어 있음 (`배송이`, `좋아요`) |
| scikit-learn 기본 | 2글자 이상 | **한 글자 단어(`안`, `차`, `솔`)가 사라짐** |

In [10]:
# 데이터의 첫 번째 리뷰를 예제로 사용
sample = work["원본텍스트"][0]
print("원문:", sample)

# 띄어쓰기로 나누기
print("띄어쓰기:", sample.split())

# scikit-learn 기본 방식으로 나누기
print("scikit-learn 기본:", CountVectorizer().build_analyzer()(sample))

원문: 기존 보유중인 55인치보다 작네요 사이즈 무슨일인가요..?;;;;
블루투스 작동 되지않아서 황당해요
띄어쓰기: ['기존', '보유중인', '55인치보다', '작네요', '사이즈', '무슨일인가요..?;;;;', '블루투스', '작동', '되지않아서', '황당해요']
scikit-learn 기본: ['기존', '보유중인', '55인치보다', '작네요', '사이즈', '무슨일인가요', '블루투스', '작동', '되지않아서', '황당해요']


---
## 3. 정규표현식으로 텍스트 정제
순서가 중요합니다. **개인정보를 먼저 가리고**, 그다음 특수문자를 지웁니다. (특수문자 `@`, `-`를 먼저 지우면 이메일·전화번호를 찾을 수 없음)

In [11]:
# 개인정보 가리기 → 특수문자 지우기 → 공백 정리 함수
def clean_text(text):
    # 이메일을 [EMAIL]로 바꾸기
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", " EMAILTOKEN ", text)
    # 휴대폰 번호를 [PHONE]으로 바꾸기
    text = re.sub(r"01[016789][- ]?\d{3,4}[- ]?\d{4}", " PHONETOKEN ", text)
    # 한글·영문·숫자·공백만 남기고 나머지는 공백으로
    text = re.sub(r"[^가-힣A-Za-z0-9\s]", " ", text)
    # 여러 칸 공백을 한 칸으로
    text = re.sub(r"\s+", " ", text).strip()
    # 임시 표시를 보기 좋은 표시로
    return text.replace("EMAILTOKEN", "[EMAIL]").replace("PHONETOKEN", "[PHONE]")

# 모든 리뷰에 적용
work["정제텍스트"] = work["원본텍스트"].apply(clean_text)
# .apply(clean_text) 각 원소를 clean_text에 하나씩 넣어 호출하고, 반환값들을 모아 새 값으로 만듦

# 원문과 비교 (앞 5개)
work[["원본텍스트", "정제텍스트"]].head()

,원본텍스트,정제텍스트
0,기존 보유중인 55인치보다 작네요 사이즈 무슨일인가요..?;;;;\n블루투스 작동 ...,기존 보유중인 55인치보다 작네요 사이즈 무슨일인가요 블루투스 작동 되지않아서 황당해요
1,크기가 55인치라 거실에 두고 사용하니 온 가족이 볼 수 있게 딱 적당하네요~\n사...,크기가 55인치라 거실에 두고 사용하니 온 가족이 볼 수 있게 딱 적당하네요 사운드...
2,해피콜부터 배송이 그렇게 빠른건아니지만 설치기사님들 친절하시고\n설명잘해주시고 가셔...,해피콜부터 배송이 그렇게 빠른건아니지만 설치기사님들 친절하시고 설명잘해주시고 가셔서...
3,방송보고 주문한지 꽤 됐는데 공사일정으로 이제 설치 받았어요~\n설치기사님이 너무너...,방송보고 주문한지 꽤 됐는데 공사일정으로 이제 설치 받았어요 설치기사님이 너무너무 ...
4,55인치쓰다가 65인치로바꾸니 좀 많이커요 ㅎㅎㅎ\n그리고 매뉴에서 각종 어플싸이크...,55인치쓰다가 65인치로바꾸니 좀 많이커요 그리고 매뉴에서 각종 어플싸이크들어갈수있...


---
## 4. 표기 통일 (공통 규칙 + 파일별 규칙)
같은 뜻인데 다르게 쓴 말을 **하나로 맞추면** 단어를 셀 때 정확해집니다.
1-3에서 고른 **파일별 규칙(`cfg`)** 과 **공통 규칙(`COMMON`)** 을 합쳐서 씁니다.

In [13]:
# 공통 규칙과 파일별 규칙 합치기
rules = {**COMMON["replace_rules"], **cfg["replace_rules"]} # **딕셔너리의 키-값 쌍을 풀어서 펼침
print("적용할 규칙:", rules)

# 표기를 통일하는 함수
def normalize_text(text):
    # 익명 처리된 브랜드명(OO, ○○) 지우기
    text = re.sub(r"(?<![A-Za-z])[Oo○]{2,}(?![A-Za-z])", " ", text)
    # 규칙대로 바꾸기
    for old, new in rules.items():
        text = text.replace(old, new)
    # 띄어 쓴 '넘'을 '너무'로
    text = re.sub(r"(?<![가-힣])넘(?= )", "너무", text)
    # 영어 소문자를 대문자로 (tv → TV)
    text = text.upper()  # 문자열 안의 소문자를 모두 대문자로 바꿈 (한글·숫자·기호는 그대로 둠)
    # 공백 정리
    return re.sub(r"\s+", " ", text).strip()

# 모든 리뷰에 적용
work["정규화텍스트"] = work["정제텍스트"].apply(normalize_text)

# 바뀐 리뷰만 골라서 개수와 예시 보기
changed = work[work["정제텍스트"] != work["정규화텍스트"]]
print("표기가 바뀐 리뷰 수:", len(changed))
changed[["정제텍스트", "정규화텍스트"]].head()

적용할 규칙: {'스크레치': '스크래치', '재 구매': '재구매', '읍니다': '습니다', '티비': 'TV'}
표기가 바뀐 리뷰 수: 1286


,정제텍스트,정규화텍스트
1,크기가 55인치라 거실에 두고 사용하니 온 가족이 볼 수 있게 딱 적당하네요 사운드...,크기가 55인치라 거실에 두고 사용하니 온 가족이 볼 수 있게 딱 적당하네요 사운드...
2,해피콜부터 배송이 그렇게 빠른건아니지만 설치기사님들 친절하시고 설명잘해주시고 가셔서...,해피콜부터 배송이 그렇게 빠른건아니지만 설치기사님들 친절하시고 설명잘해주시고 가셔서...
5,티비가 고장나서 사는김에 좀 큰 65인치로 샀는데 잘 산거 같아요 처음에는 커보이더...,TV가 고장나서 사는김에 좀 큰 65인치로 샀는데 잘 산거 같아요 처음에는 커보이더...
6,적립금 속았어요 바로 적립되는것도 아니고 다른 홈쇼핑은 적립됐다 소멸된다 알려주는데...,적립금 속았어요 바로 적립되는것도 아니고 다른 홈쇼핑은 적립됐다 소멸된다 알려주는데...
8,티비 너무 만족합니다 화질이 선명하고 좋습니다 사이즈도 커서 완전 만족이네요 설치전...,TV 너무 만족합니다 화질이 선명하고 좋습니다 사이즈도 커서 완전 만족이네요 설치전...


---
## 5. 한국어 형태소 분석과 scikit-learn 연결

### 5-1. 사용자 사전 등록 (파일별)
분석기는 처음 보는 합성어를 **잘못 나눌** 수 있습니다. 1-3 설정의 `user_words`를 등록하면 한 단어로 분석합니다.

In [14]:
# 사용자 사전이 없는 분석기와 있는 분석기 만들기
kiwi_plain = Kiwi()
kiwi = Kiwi()

# 파일별 사용자 단어를 일반 명사(NNG)로 등록
for word in cfg["user_words"]:
    kiwi.add_user_word(word, "NNG")

# 등록 전후 비교
for word in cfg["user_words"][:4]:
    before = [t.form for t in kiwi_plain.tokenize(word)]
    after = [t.form for t in kiwi.tokenize(word)]
    print(f"{word}: 등록 전 {before} → 등록 후 {after}")

사운드바: 등록 전 ['사운드', '바'] → 등록 후 ['사운드바']
미러링: 등록 전 ['미러', '링'] → 등록 후 ['미러링']
베젤: 등록 전 ['베', '젤'] → 등록 후 ['베젤']
중저음: 등록 전 ['중', '저', '음'] → 등록 후 ['중저음']


### 5-2. 필요한 단어만 고르기
| 설정 | 내용 |
|---|---|
| 남길 품사 | 일반명사 `NNG`, 고유명사 `NNP`, 동사 `VV`, 형용사 `VA`, 어근 `XR`(깔끔·튼튼·웅장), 영어 `SL`(TV·GPS·USB) |
| 꼭 남길 단어 | 부정·평가 표현 `안`, `못`, `않다`, `없다`, `별로` |
| 뺄 단어 | 공통 불용어 + **파일별 불용어** |
| 표시 모양 | 동사·형용사는 `○○다`, 어근은 `○○하다` (예: `튼튼` → `튼튼하다`) |

In [25]:
# 남길 품사 일반 명사,고유 명사, 동사, 	형용사,어근, 알파벳
keep_tags = {"NNG",    "NNP",    "VV",  "VA",  "XR", "SL"}
# 품사와 관계없이 꼭 남길 단어
keep_words = {"안", "못", "않다", "없다", "별로"}
# 뺄 단어 = 공통 불용어 + 파일별 불용어
stop_words = COMMON["stop_words"] | cfg["stop_words"]

# 형태소 분석 결과에서 필요한 단어만 골라 목록으로 돌려주는 함수
def select_tokens(tokens):
    words = []
    for t in tokens:
        tag = t.tag               # 'VV' 같은 태그를 '
        if tag in {"VV", "VA", "VX"}:
            word = t.form + "다"                 # 동사·형용사는 '다' 붙이기
        elif tag == "XR":     #어근
            word = t.form + "하다"               # 어근은 '하다' 붙이기
        else:
            word = t.form
        # 뺄 단어가 아니고, 꼭 남길 단어이거나 남길 품사이면 추가
        if word not in stop_words and (word in keep_words or tag in keep_tags):
            words.append(word)
    return words

# 첫 리뷰로 확인
print("원문:", work["정규화텍스트"][0])
print("형태소:", [(t.form, t.tag) for t in kiwi.tokenize(work["정규화텍스트"][0])])
print("고른 단어:", select_tokens(kiwi.tokenize(work["정규화텍스트"][0])))

원문: 기존 보유중인 55인치보다 작네요 사이즈 무슨일인가요 블루투스 작동 되지않아서 황당해요
형태소: [('기존', 'NNG'), ('보유', 'NNG'), ('중', 'NNB'), ('이', 'VCP'), ('ᆫ', 'ETM'), ('55', 'SN'), ('인치', 'NNB'), ('보다', 'JKB'), ('작', 'VA'), ('네요', 'EF'), ('사이즈', 'NNG'), ('무슨', 'MM'), ('일', 'NNG'), ('이', 'VCP'), ('ᆫ가요', 'EF'), ('블루투스', 'NNP'), ('작동', 'NNG'), ('되', 'XSV'), ('지', 'EC'), ('않', 'VX'), ('어서', 'EC'), ('황당하', 'VA'), ('어요', 'EF')]
고른 단어: ['기존', '보유', '작다', '사이즈', '일', '블루투스', '작동', '않다', '황당하다']


### 5-3. scikit-learn에 연결하고 전체 리뷰 분석

In [26]:
# 문장 하나를 넣으면 고른 단어 목록을 돌려주는 함수 (scikit-learn 연결용)
def kiwi_tokenizer(text):
    return select_tokens(kiwi.tokenize(text))

# Kiwi 토크나이저를 연결한 CountVectorizer
vectorizer = CountVectorizer(tokenizer=kiwi_tokenizer, token_pattern=None, lowercase=False)
print("CountVectorizer 결과:", vectorizer.build_analyzer()(work["정규화텍스트"][0]))

# 전체 리뷰를 한 번에 분석 (여러 문장을 넣으면 더 빠름, 10~30초)
work["분석토큰"] = [select_tokens(t) for t in kiwi.tokenize(work["정규화텍스트"].tolist())]
work[["정규화텍스트", "분석토큰"]].head()

CountVectorizer 결과: ['기존', '보유', '작다', '사이즈', '일', '블루투스', '작동', '않다', '황당하다']


,정규화텍스트,분석토큰
0,기존 보유중인 55인치보다 작네요 사이즈 무슨일인가요 블루투스 작동 되지않아서 황당해요,"[기존, 보유, 작다, 사이즈, 일, 블루투스, 작동, 않다, 황당하다]"
1,크기가 55인치라 거실에 두고 사용하니 온 가족이 볼 수 있게 딱 적당하네요 사운드...,"[크기, 거실, 두다, 가족, 보다, 적당하다, 사운드, 바, 소리, 영화관, 오다]"
2,해피콜부터 배송이 그렇게 빠른건아니지만 설치기사님들 친절하시고 설명잘해주시고 가셔서...,"[해피, 콜, 배송, 빠르다, 설치, 기사, 친절, 설명, 가다, 만족, 기존, 크..."
3,방송보고 주문한지 꽤 됐는데 공사일정으로 이제 설치 받았어요 설치기사님이 너무너무 ...,"[방송, 보다, 주문, 공사, 일정, 설치, 설치, 기사, 친절, 벽걸이, 앞, 뒤..."
4,55인치쓰다가 65인치로바꾸니 좀 많이커요 그리고 매뉴에서 각종 어플싸이크들어갈수있...,"[쓰다, 바꾸다, 크다, 매뉴, 각종, 어플싸이크, 들어가다, 좋다, 애, 인터넷,..."


### 5-4. 자주 나온 단어 확인 (설정 고치기)
상위 단어를 보고 **의미 없이 많이 나오는 단어**가 있으면 1-3의 `stop_words`에, **이상하게 쪼개진 단어**가 있으면 `user_words`에 넣고 **1-3 셀부터 다시 실행**하세요.

In [27]:
# 단어 세기 도구
from collections import Counter

# 모든 리뷰의 단어를 세어 상위 20개 출력
counts = Counter(word for words in work["분석토큰"] for word in words)
counts.most_common(20)

[('좋다', 6621),
 ('만족', 3085),
 ('가격', 1405),
 ('화질', 1268),
 ('선명', 1233),
 ('소리', 1113),
 ('디자인', 1010),
 ('크다', 1001),
 ('배송', 998),
 ('음질', 950),
 ('들다', 949),
 ('저렴하다', 935),
 ('보다', 902),
 ('연결', 831),
 ('설치', 827),
 ('없다', 803),
 ('기능', 767),
 ('사운드', 732),
 ('굿', 714),
 ('않다', 662)]

---
## 6. 결과 저장
단어 목록을 띄어쓰기로 이어서 `토큰문자열` 열로 저장합니다. 원문·정제·정규화 텍스트도 함께 남겨야 **어떻게 가공했는지 추적**할 수 있습니다.

In [28]:
# 단어 목록을 띄어쓰기로 이어 붙이기
work["토큰문자열"] = work["분석토큰"].apply(" ".join)

# 저장할 열만 고르기
output = work[["번호", "원본텍스트", "정제텍스트", "정규화텍스트", "토큰문자열"]]

# 엑셀 파일 이름으로 저장 파일 이름 만들기 (예: cleaned_TS_5-4.청소세탁용품.csv)
path = "cleaned_" + excel_path.rsplit(".", 1)[0] + ".csv"

# 한글이 깨지지 않게 utf-8-sig로 저장
output.to_csv(path, index=False, encoding="utf-8-sig")
print("저장 완료:", path, "/", len(output), "건")
output.head()

저장 완료: cleaned_TS_3-1.영상음향가전.csv / 6001 건


,번호,원본텍스트,정제텍스트,정규화텍스트,토큰문자열
0,2,기존 보유중인 55인치보다 작네요 사이즈 무슨일인가요..?;;;;\n블루투스 작동 ...,기존 보유중인 55인치보다 작네요 사이즈 무슨일인가요 블루투스 작동 되지않아서 황당해요,기존 보유중인 55인치보다 작네요 사이즈 무슨일인가요 블루투스 작동 되지않아서 황당해요,기존 보유 작다 사이즈 일 블루투스 작동 않다 황당하다
1,3,크기가 55인치라 거실에 두고 사용하니 온 가족이 볼 수 있게 딱 적당하네요~\n사...,크기가 55인치라 거실에 두고 사용하니 온 가족이 볼 수 있게 딱 적당하네요 사운드...,크기가 55인치라 거실에 두고 사용하니 온 가족이 볼 수 있게 딱 적당하네요 사운드...,크기 거실 두다 가족 보다 적당하다 사운드 바 소리 영화관 오다
2,4,해피콜부터 배송이 그렇게 빠른건아니지만 설치기사님들 친절하시고\n설명잘해주시고 가셔...,해피콜부터 배송이 그렇게 빠른건아니지만 설치기사님들 친절하시고 설명잘해주시고 가셔서...,해피콜부터 배송이 그렇게 빠른건아니지만 설치기사님들 친절하시고 설명잘해주시고 가셔서...,해피 콜 배송 빠르다 설치 기사 친절 설명 가다 만족 기존 크다 몰입 감 좋다 사운...
3,5,방송보고 주문한지 꽤 됐는데 공사일정으로 이제 설치 받았어요~\n설치기사님이 너무너...,방송보고 주문한지 꽤 됐는데 공사일정으로 이제 설치 받았어요 설치기사님이 너무너무 ...,방송보고 주문한지 꽤 됐는데 공사일정으로 이제 설치 받았어요 설치기사님이 너무너무 ...,방송 보다 주문 공사 일정 설치 설치 기사 친절 벽걸이 앞 뒤 좌우 상하 움직이다 ...
4,6,55인치쓰다가 65인치로바꾸니 좀 많이커요 ㅎㅎㅎ\n그리고 매뉴에서 각종 어플싸이크...,55인치쓰다가 65인치로바꾸니 좀 많이커요 그리고 매뉴에서 각종 어플싸이크들어갈수있...,55인치쓰다가 65인치로바꾸니 좀 많이커요 그리고 매뉴에서 각종 어플싸이크들어갈수있...,쓰다 바꾸다 크다 매뉴 각종 어플싸이크 들어가다 좋다 애 인터넷 방송 편하다


---
## 정리

### ✅ 한 줄 정리
**코드는 하나, 설정은 파일마다.** 데이터가 바뀌면 `replace_rules`, `user_words`, `stop_words`를 그 데이터에 맞게 고칩니다.

### 🤔 생각해 볼 점
- **표기 통일의 기준:** `네비`와 `내비` 중 무엇으로 맞출지는 정하기 나름입니다. 이 실습은 표준어(`내비게이션`) 쪽으로 맞췄습니다. 중요한 것은 **한 가지로 통일**하는 것입니다.
- **불용어는 목적에 따라:** 자동차기기에서 `차`를 뺐지만, "차량 호환" 문제를 분석한다면 `차`가 중요한 단어일 수 있습니다.
- **사용자 사전의 부작용:** `사운드바`를 등록하면 `바`(막대)가 따로 쓰인 문장에는 영향이 없지만, 단어를 너무 많이 등록하면 다른 문장이 잘못 분석될 수 있습니다. 등록 후 결과를 꼭 확인하세요.
- **같은 단어, 다른 의미:** `크다`, `작다`는 TV에서는 화면 크기, 블랙박스에서는 기기 크기를 뜻합니다. 데이터마다 해석이 달라집니다.



**참고 공식 문서**
- [scikit-learn CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- [Kiwi (kiwipiepy)](https://github.com/bab2min/kiwipiepy)